<a href="https://colab.research.google.com/github/Salokin6/Salonki/blob/main/lego_collab_(Nikolas_Alasm%C3%A4ki).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Asennetaan YOLOv8
!pip install ultralytics --quiet

# Tuodaan tarvittavat kirjastot
from ultralytics import YOLO
import os
from google.colab import files


In [ ]:
# Luo tarvittavat kansiot
os.makedirs("data/legos/spiderman", exist_ok=True)
os.makedirs("data/legos/normal", exist_ok=True)

# Spiderman-legojen lataus
print("📥 Lataa nyt Spiderman-lego kuvia:")
uploaded_spider = files.upload()
for fn in uploaded_spider.keys():
    os.rename(fn, f"data/legos/spiderman/{fn}")

# Tavallisten legoukkojen lataus
print("📥 Lataa nyt tavallisten legoukkojen kuvia:")
uploaded_normal = files.upload()
for fn in uploaded_normal.keys():
    os.rename(fn, f"data/legos/normal/{fn}")


In [ ]:
# Lataa valmis YOLOv8-luokituspohja. Käytän itsekin nanomallia.
model = YOLO("yolov8n-cls.pt")

# Mallin koulutus. Kuvien koko muutettu 64 > 224, jotta tarkemmat kuvat.
model.train(data="data/legos", epochs=10, imgsz=224)


In [ ]:
# Lataa testikuva
print("📥 Lataa testikuva:")
test_img = files.upload()
img_path = list(test_img.keys())[0]

# Aja mallilla ja ennusta
results = model(img_path)

# Tulkitse tulos
probs = results[0].probs
predicted_class = probs.top1
predicted_prob = probs.top1conf.item()
predicted_name = results[0].names[predicted_class]

# Alla oleva koodi tekee kolmannen tulosten tulkinnan ja ilmoittaa kuvaksi "Muu", jos se ei tunnista kuvaa Spider tai normaaliksi legoksi.
kynnysarvo = 0.70

# Tarkistetaan, onko malli tarpeeksi varma.
if predicted_prob < kynnysarvo:
    print(f"\n Tulos: Muu")
    print(f"(Malli ei ollut tarpeeksi varma. Lähin arvaus oli {predicted_name.upper()} {predicted_prob:.2%} varmuudella.)")
else:
    print(f"\n Kuvassa on: {predicted_name.upper()} ({predicted_prob:.2%} varmuudella)")
